# 02 DQN 实战：CartPole-v1

**目标**：从零写一个 DQN，在 CartPole-v1 上达到 ≥ 195 平均回报（满分 500）。

**主要组件**：
1. Q 网络（3 层 MLP）
2. Replay Buffer
3. Target Network
4. ε-greedy + 衰减

In [ ]:
import random
from collections import deque
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
class QNet(nn.Module):
    def __init__(self, obs_dim, n_actions, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_actions)
        )
    def forward(self, x):
        return self.net(x)

class ReplayBuffer:
    def __init__(self, capacity=50_000):
        self.buf = deque(maxlen=capacity)
    def push(self, s, a, r, s2, d):
        self.buf.append((s, a, r, s2, d))
    def sample(self, batch_size):
        batch = random.sample(self.buf, batch_size)
        s, a, r, s2, d = zip(*batch)
        return (torch.tensor(np.array(s), dtype=torch.float32),
                torch.tensor(a, dtype=torch.long),
                torch.tensor(r, dtype=torch.float32),
                torch.tensor(np.array(s2), dtype=torch.float32),
                torch.tensor(d, dtype=torch.float32))
    def __len__(self):
        return len(self.buf)

In [ ]:
ENV_ID = 'CartPole-v1'
GAMMA = 0.99
LR = 1e-3
BATCH_SIZE = 64
BUFFER_CAP = 50_000
WARMUP = 1_000
TARGET_UPDATE_FREQ = 500     
EPS_START, EPS_END, EPS_DECAY = 1.0, 0.05, 10_000
MAX_STEPS = 30_000

env = gym.make(ENV_ID)
obs_dim = env.observation_space.shape[0]
n_actions = env.action_space.n

q_net = QNet(obs_dim, n_actions).to(device)
target_net = QNet(obs_dim, n_actions).to(device)
target_net.load_state_dict(q_net.state_dict())
optimizer = optim.Adam(q_net.parameters(), lr=LR)
buffer = ReplayBuffer(BUFFER_CAP)

def epsilon(step):
    return max(EPS_END, EPS_START - (EPS_START - EPS_END) * step / EPS_DECAY)

def select_action(obs, step):
    if random.random() < epsilon(step):
        return env.action_space.sample()
    with torch.no_grad():
        q = q_net(torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0))
    return q.argmax(-1).item()

In [ ]:
obs, _ = env.reset(seed=SEED)
ep_return = 0
returns_history = []

for global_step in range(1, MAX_STEPS + 1):
    a = select_action(obs, global_step)
    obs2, r, term, trunc, _ = env.step(a)
    done = term or trunc
    buffer.push(obs, a, r, obs2, float(term))   
    obs = obs2
    ep_return += r

    if done:
        returns_history.append(ep_return)
        obs, _ = env.reset()
        ep_return = 0

    if len(buffer) >= max(WARMUP, BATCH_SIZE):
        s, a_b, r_b, s2, d_b = buffer.sample(BATCH_SIZE)
        s, a_b, r_b, s2, d_b = s.to(device), a_b.to(device), r_b.to(device), s2.to(device), d_b.to(device)
        q_pred = q_net(s).gather(1, a_b.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            q_next = target_net(s2).max(-1).values
            target = r_b + GAMMA * (1 - d_b) * q_next
        loss = ((q_pred - target) ** 2).mean()
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(q_net.parameters(), 10.0)
        optimizer.step()

    if global_step % TARGET_UPDATE_FREQ == 0:
        target_net.load_state_dict(q_net.state_dict())

    if global_step % 2000 == 0 and len(returns_history) > 0:
        recent = np.mean(returns_history[-20:])
        print(f'step={global_step:6d}  eps={epsilon(global_step):.3f}  recent20_return={recent:.1f}')

env.close()

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(returns_history, alpha=0.3, label='episode return')
if len(returns_history) > 20:
    smooth = np.convolve(returns_history, np.ones(20)/20, mode='valid')
    plt.plot(np.arange(19, len(returns_history)), smooth, label='smoothed (20)', linewidth=2)
plt.axhline(195, color='r', linestyle='--', label='solved')
plt.xlabel('Episode'); plt.ylabel('Return'); plt.legend(); plt.grid(True)
plt.title('DQN on CartPole-v1')
plt.show()

## 测试训练好的 agent

贪心策略，看能撑多少步。

In [ ]:
test_env = gym.make(ENV_ID)
scores = []
for ep in range(10):
    obs, _ = test_env.reset(seed=ep)
    total = 0
    while True:
        with torch.no_grad():
            a = q_net(torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)).argmax(-1).item()
        obs, r, term, trunc, _ = test_env.step(a)
        total += r
        if term or trunc:
            break
    scores.append(total)
test_env.close()
print('test scores:', scores)
print('mean:', np.mean(scores))

## 练习

1. 改成 **Double DQN**：把 target 改为 `Q_target(s', argmax_a Q_online(s', a))`，看是否更稳定。
2. 在 LunarLander-v2 上跑（动作数=4，状态=8），需要更大的网络/更多步。
3. 把 ε-greedy 换成 NoisyNet（参数空间扰动）。
4. 加 TensorBoard 记录 loss / Q-mean / epsilon。